# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:29<00:00, 17.82s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

'Title: CanKeeper 3-in-1 Double Walled Beverage Insulator for $5 + free shipping\nDetails: It\'s $35 off. We\'ve pictured it in Black, but it\'s available in several colors. Plus, apply code "DEALNEWS" to get free shipping, an additional $8.99 savings. Buy Now at MorningSave\nFeatures: double-walled and vacuum insulated sweat-proof exterior measures 7.38" H x 3.25" W x 3.25" D\nURL: https://www.dealnews.com/Can-Keeper-3-in-1-Double-Walled-Beverage-Insulator-for-5-free-shipping/21729984.html?iref=rss-c196'

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Kodak Mini 2 Retro Bundle for $90 in cart + free shipping
Details: The price for this bundle drops once you add it to your cart — it's a $50 savings in the end. Buy Now at Kodak Photo Printer (Prinics America)
Features: Kodak Mini 2 Retro portable photo printer photo cartridge for 60 photos Model: P210R
URL: https://www.dealnews.com/products/Kodak/Kodak-Mini-2-Retro

In [9]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [10]:
result = get_recommendations()

In [11]:
len(result.deals)

5

In [12]:
result.deals[1]

Deal(product_description='The Eco-Worthy Server Rack LiFePO4 Lithium Battery 4-Pack provides reliable power with a total capacity of 20.48kWh and 48V 100AH configuration. Designed for standard 3U cabinets, these batteries offer a long lifespan with up to 6,000 deep discharge cycles, making them suitable for various applications. They also come with wireless connectivity and smart monitoring features, ensuring an efficient and modern power solution.', price=3099.0, url='https://www.dealnews.com/Eco-Worthy-20-48-k-Wh-48-V-100-AH-Server-Rack-Li-Fe-PO4-Lithium-Battery-4-Pack-for-3-099-free-shipping/21729917.html?iref=rss-c142')

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

In [15]:
result

DealSelection(deals=[Deal(product_description='The Insignia 85" Class F50 Series LED 4K UHD Smart Fire TV offers an immersive viewing experience with its expansive screen, providing vivid colors and sharp details. This model integrates with Amazon\'s Fire TV platform, allowing access to popular streaming services and Alexa voice control. With advanced picture processing, it enhances contrast and brightness, ensuring high-quality visuals for all types of media. Ideal for movie nights or binge-watching your favorite series, this TV also features multiple HDMI inputs for flexible connectivity.', price=600.0, url='https://www.dealnews.com/products/Insignia/Insignia-85-Class-F50-Series-LED-4-K-UHD-Smart-Fire-TV/488745.html?iref=rss-c142'), Deal(product_description='The Kodak Mini 2 Retro Bundle is a compact, portable photo printer that combines modern technology with a nostalgic design. It allows you to instantly print high-quality photos from your smartphone or tablet. This bundle includes